In [1]:
# 모델 사용 시 알아야 할 것:
# 모델 사용하기에서 실제 값(URL 등)을 확인
# 코드에 넣었을 때 안 되거나 헷갈리는 부분이 있으면 (baseUrl에 v1 등이 필요할 수도 있음)
# AI에게 스크린샷이나 에러 메시지 보여주면서 물어보기
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv("../.env")
API_KEY = os.getenv("OPEN_API_KEY")
BASE_URL = os.getenv("MLAPI_BASE_URL")

# Elice AI Cloud에서 발급받은 키 사용시
def get_response(prompt: str) -> str:
  client = OpenAI(
    base_url=BASE_URL, # e.g. "https://mlapi.run/.../v1"
    api_key=API_KEY
  )
  resp = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
  )
  msg = resp.choices[0].message # 응답으로부터 메시지를 추출
  return msg


if __name__ == "__main__":
  prompt = input("You: ")
  response = get_response(prompt)
  print(f"AI:", response)

AI: ChatCompletionMessage(content='안녕하세요! 저는 잘 지내고 있습니다. 당신은 어떠세요?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [ ]:
# 시스템 프롬프트 — "너는 쇼핑마트의 AI 챗봇이야... 상품 목록에서 선별해서 추천해야 해"라는 역할 설정
# 네가 입력한 프롬프트 — 실제 질문 내용
# 함수(tool)의 description — find_products_info에 적힌 "제품 설명에 keywords가 포함된 제품을 찾아서 반환합니다..."라는 설명

# 이 세 가지를 모델이 한꺼번에 보고, "지금 이 질문이 저 함수랑 관련 있나?"를 API 호출 시점마다 스스로 판단(관련 있다면 함수 호출 없다면 일반 대답)
import json
import sys
from pprint import pprint
import os
from dotenv import load_dotenv

load_dotenv("../.env")
API_KEY = os.getenv("OPENAI_API_KEY")
BASE_URL = os.getenv("MLAPI_BASE_URL")

sys.stdin.reconfigure(encoding="utf-8")

import pandas as pd
from openai import OpenAI
from openai.types.chat import ChatCompletionMessage

# 1. OpenAI 웹사이트에서 발급받은 키 사용시
# client = OpenAI(api_key="sk-")

# 2. Elice AI Cloud에서 발급받은 키 사용시
client = OpenAI(
  base_url=BASE_URL,
  api_key=API_KEY
)


def find_products(keywords: list[str]) -> list[dict]:
  """제품 설명에 keywords가 포함된 제품을 찾아서 반환합니다."""

  print("find_products 함수", keywords)
  df = pd.read_csv("product_list.csv")
  result = df[df["description"].str.contains("|".join(keywords))]
  return result.to_dict(orient="records")


def get_response(chat_log: list, tools: dict = None) -> ChatCompletionMessage:
  # 1. OpenAI 웹사이트에서 발급받은 키 사용시
  # response = client.chat.completions.create(model="gpt-4o-mini", messages=chat_log, tools=tools)

  # 2. Elice AI Cloud에서 발급받은 키 사용시
  response = client.chat.completions.create(model="openai/gpt-4o-mini", messages=chat_log, tools=tools)

  if response.choices[0].message.tool_calls:
    tool_call_results = []
    for tool_call in response.choices[0].message.tool_calls:
      call_id = tool_call.id

      if tool_call.function.name == "find_products":
        keywords = json.loads(tool_call.function.arguments)["keywords"]
        products = find_products(keywords)

        call_result_message = {
          "tool_call_id": call_id,
          "role": "tool",
          "content": json.dumps(
            {"keywords": keywords, "products": str(products)},
            ensure_ascii=False,
          ),
        }
        tool_call_results.append(call_result_message)

    messages = chat_log + [response.choices[0].message] + tool_call_results

    # 1. OpenAI 웹사이트에서 발급받은 키 사용시
    # response = client.chat.completions.create(
    #   model="gpt-4o-mini",
    #   messages=messages,
    #   tools=tools,
    # )

    # 2. Elice AI Cloud에서 발급받은 키 사용시
    response = client.chat.completions.create(
      model="openai/gpt-4o-mini",
      messages=messages,
      tools=tools,
    )

  return response.choices[0].message


find_products_info = {
  "name": "find_products",
  "description": "제품 설명에 keywords가 포함된 제품을 찾아서 반환합니다. 찾고자 하는 keywords는 사용자가 요청한 내용과 관련된 제품을 필터링할 수 있는 적절한 키워드여야 합니다.",
  "parameters": {
    "type": "object",
    "properties": {
      "keywords": {
        "type": "array",
        "items": {"type": "string"},
        "description": "상품 목록에서 검색할 키워드들. 이 키워드는 한글로 구성되어 있고 하나라도 포함된 상품을 모두 가져오기에 적절한 키워드만 전달해야 합니다.",
      },
    },
    "required": ["keywords"],
    "additionalProperties": False,
  },
}

tools = [{"type": "function", "function": find_products_info}]


def main():
  chat_log = [
    {
      "role": "system",
      "content": "너는 쇼핑마트의 AI 챗봇이야. 다양한 제품에 대해 물어보거나 추천을 하는 용도야. 추천을 할 때는 상품 목록에서 사용자가 원하는 상품만 선별해서 추천해야 해.",
    },
  ]

  chat_log.append({
    "role": "user",
    "content": input("You> ")
  })
  response = get_response(chat_log, tools=tools)
  print("AI>", response.content)


if __name__ == "__main__":
  main()

find_products 함수 ['다이어트', '식품', '알러지']
AI> 견과류 알러지가 있는 분들을 위해 안전한 다이어트 식품을 추천드립니다:

1. **다이어트 샐러드B**
   - 가격: 9,000원
   - 설명: 완제품 샐러드, 다이어트 식품, 아몬드 대신 연어가 추가된 제품.
   - 재고: 5개

2. **다이어트 샐러드C**
   - 가격: 8,000원
   - 설명: 완제품 샐러드, 다이어트 식품, 아몬드 대신 닭가슴살이 추가된 제품.
   - 재고: 0개 (현재 품절)

3. **바나나 프로틴쉐이크**
   - 가격: 5,500원
   - 설명: 바나나 맛 단백질 쉐이크, 350ml, 다이어트 식품.
   - 재고: 10개

4. **닭가슴살 스낵**
   - 가격: 4,000원
   - 설명: 고단백 저지방 닭가슴살 스낵, 50g, 다이어트 간식.
   - 재고: 20개

5. **퀴노아 샐러드**
   - 가격: 9,000원
   - 설명: 퀴노아와 야채가 가득한 샐러드, 다이어트 식품.
   - 재고: 5개

이 중에서 관심 있는 제품에 대해 더 알아보시겠어요?
